<!-- paired-role-banner -->
> 🟩 **정답 노트북 05 · 참고 코드 창**
>
> 왼쪽 실습 노트북과 같은 셀 순서입니다. 결과만 복사하기보다 각 shape, 자료형, 손실값과
> 설계 이유를 먼저 예측한 뒤 필요한 부분만 확인하세요.

# 05. PyTorch 핵심: Tensor · Autograd · Device · 디버깅

이 노트북의 목표는 API 암기가 아니라 **텐서가 모델 안에서 어떻게 이동하고 미분되는지 추적하는 습관**을 만드는 것입니다.

예상 시간: 60~90분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 선행 지식: Python 함수/클래스, NumPy 기초

> 권장 방식: 각 셀을 그대로 한 번 실행한 뒤, 숫자·shape·축을 바꾸어 다시 실행하세요. 오류 메시지를 읽는 과정도 실습입니다.

## 이번 노트북에서 익힐 것

- `shape`, `dtype`, `device`를 데이터 계약(data contract)으로 읽기
- broadcasting과 차원 변환을 작은 예제로 확인하기
- autograd 그래프, leaf tensor, gradient 누적을 이해하기
- `no_grad`, `inference_mode`, `detach`의 역할 구분하기
- NaN/Inf, 잘못된 shape, 끊어진 gradient를 빠르게 진단하기
- 작은 최적화 루프를 직접 작성하고 수치 미분으로 검산하기

In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
assert PROJECT_ROOT is not None, "프로젝트 폴더 안에서 노트북을 열어 주세요."
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from llm_engineering_lab.acceleration import get_accelerator

ACCELERATOR = get_accelerator()

print("project:", PROJECT_ROOT)
print("torch:", torch.__version__)
print(ACCELERATOR.summary())

## 1. Tensor는 값 + 메타데이터다

텐서를 볼 때는 값만 보지 말고 다음 네 가지를 함께 확인합니다.

1. `shape`: 각 축의 의미는 무엇인가?
2. `dtype`: 정수 ID인가, 실수 특징인가?
3. `device`: CPU와 GPU 중 어디에 있는가?
4. `requires_grad`: 이 값까지 gradient를 계산해야 하는가?

LLM에서 흔한 표기는 `[B, T, D] = [batch, time, hidden]`입니다.

In [ ]:
token_ids = torch.tensor([[2, 10, 11, 3], [2, 20, 3, 0]], dtype=torch.long)
embeddings = torch.randn(2, 4, 8, dtype=torch.float32)

for name, tensor in {"token_ids": token_ids, "embeddings": embeddings}.items():
    print(
        f"{name:12s} shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}, device={tensor.device}"
    )

assert token_ids.shape == (2, 4)  # [B, T]
assert embeddings.shape == (2, 4, 8)  # [B, T, D]
assert token_ids.dtype == torch.long

In [ ]:
# dtype에 따라 가능한 연산이 달라집니다.
ids = torch.tensor([1, 2, 3])
values = torch.tensor([1.0, 2.0, 3.0])
mask = torch.tensor([True, False, True])

print(ids.dtype, values.dtype, mask.dtype)
print("정수 -> 실수:", ids.float())
print("마스크로 선택:", values[mask])

embedding = nn.Embedding(num_embeddings=10, embedding_dim=4)
embedded = embedding(ids)  # Embedding의 인덱스는 torch.long이어야 합니다.
print("embedded shape:", embedded.shape)
assert embedded.shape == (3, 4)

## 2. Shape를 바꾸는 연산

- `unsqueeze`: 길이 1인 축 추가
- `squeeze`: 길이 1인 축 제거
- `reshape`: 원소 수를 유지하며 모양 변경
- `transpose`: 두 축 교환
- `permute`: 전체 축 순서 재배열

`view`는 메모리가 contiguous인지에 영향을 받습니다. 학습 코드에서는 의도를 분명히 하기 위해 `reshape`를 먼저 쓰는 편이 안전합니다.

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)  # [B=2, T=3, D=4]
print("x:              ", x.shape)
print("unsqueeze(1):   ", x.unsqueeze(1).shape)
print("transpose(1, 2):", x.transpose(1, 2).shape)
print("flatten B,T:    ", x.reshape(-1, x.size(-1)).shape)

flat = x.reshape(-1, 4)
restored = flat.reshape(2, 3, 4)
assert torch.equal(x, restored)

In [ ]:
# Broadcasting: 뒤쪽 축부터 크기가 같거나, 둘 중 하나가 1이어야 합니다.
batch = torch.arange(6, dtype=torch.float32).reshape(2, 3)  # [B, D]
bias = torch.tensor([10.0, 20.0, 30.0])  # [D]
scale = torch.tensor([[1.0], [2.0]])  # [B, 1]

shifted = batch + bias
scaled = batch * scale
print("batch + bias:\n", shifted)
print("batch * scale:\n", scaled)
assert shifted.shape == scaled.shape == (2, 3)

In [ ]:
def inspect(name: str, tensor: torch.Tensor) -> None:
    print(
        f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
        f"device={tensor.device}, finite={torch.isfinite(tensor).all().item()}"
    )


a = torch.randn(2, 3, 4)
b = torch.randn(2, 4)
inspect("a", a)
inspect("b", b)

try:
    _ = a + b
except RuntimeError as error:
    print("예상한 shape 오류:", str(error).splitlines()[0])

# b를 [B, 1, D]로 바꾸면 time 축으로 broadcasting됩니다.
fixed = a + b.unsqueeze(1)
inspect("fixed", fixed)
assert fixed.shape == a.shape

In [ ]:
# TODO: 직접 다시 타이핑해 보세요 — feature별 표준화
features = torch.tensor([[1.0, 10.0, 100.0], [3.0, 14.0, 120.0], [5.0, 18.0, 140.0]])
mean = features.mean(dim=0, keepdim=True)
std = features.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
standardized = (features - mean) / std

print("mean:", standardized.mean(dim=0))
print("std: ", standardized.std(dim=0, unbiased=False))
assert torch.allclose(standardized.mean(0), torch.zeros(3), atol=1e-6)

# 실험: dim=0을 dim=1로 바꾸면 무엇을 표준화하게 될까요?

## 3. Device 관리

모델과 입력은 같은 장치에 있어야 합니다. 이 저장소는 `AI_LAB_DEVICE=auto`에서 CUDA → MPS → CPU 순으로 선택합니다. 재현성 비교가 필요하면 실행 전에 `AI_LAB_DEVICE=cpu`로 강제할 수 있습니다.

실무 팁: checkpoint는 `map_location="cpu"`로 먼저 읽고, 모델을 원하는 장치로 옮기면 이식성이 좋아집니다.

In [ ]:
device = ACCELERATOR.device

model = nn.Linear(4, 2).to(device)
inputs = torch.randn(3, 4).to(device)
outputs = model(inputs)

print("parameter device:", next(model.parameters()).device)
print("input/output:", inputs.device, outputs.device)
assert next(model.parameters()).device == inputs.device == outputs.device

## 4. Autograd: 계산 그래프를 역방향으로 따라가기

`requires_grad=True`인 leaf tensor에서 시작한 연산은 계산 그래프를 만듭니다. 최종 scalar loss에 `backward()`를 호출하면 chain rule로 각 leaf의 `.grad`가 채워집니다.

$$y = x^2 + 3x,\quad \frac{dy}{dx}=2x+3$$

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()

print("y:", y.item())
print("autograd dy/dx:", x.grad.item())
print("analytic dy/dx:", 2 * x.item() + 3)
assert torch.allclose(x.grad, torch.tensor(7.0))

In [ ]:
features = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
weights = torch.randn(2, 1, requires_grad=True)  # leaf parameter
predictions = features @ weights  # non-leaf tensor
loss = predictions.square().mean()  # scalar

print("weights: leaf=", weights.is_leaf, "grad_fn=", weights.grad_fn)
print("predictions: leaf=", predictions.is_leaf, "grad_fn=", predictions.grad_fn)
print("loss grad_fn:", loss.grad_fn)
loss.backward()
print("weights.grad shape:", weights.grad.shape)
assert weights.grad.shape == weights.shape

In [ ]:
# gradient는 기본적으로 누적됩니다. 미니배치마다 초기화해야 합니다.
w = torch.tensor(2.0, requires_grad=True)
(w**2).backward()
first = w.grad.item()
(w**2).backward()
accumulated = w.grad.item()
print("first:", first, "after second backward:", accumulated)
assert accumulated == 2 * first

w.grad = None  # optimizer.zero_grad(set_to_none=True)와 같은 의도
(w**2).backward()
print("after reset:", w.grad.item())

### `detach`, `no_grad`, `inference_mode`

- `detach()`: 특정 tensor를 현재 그래프에서 분리
- `torch.no_grad()`: 블록 안의 gradient 기록 중지
- `torch.inference_mode()`: 추론 전용으로 더 강한 최적화 적용

평가/배포 추론에는 `model.eval()`과 `inference_mode()`를 함께 사용합니다. `eval()`은 dropout·batch norm 동작을 바꾸지만 gradient 기록을 끄지는 않습니다.

In [ ]:
source = torch.tensor([1.0, 2.0], requires_grad=True)
tracked = source * 3
detached = tracked.detach()

with torch.no_grad():
    no_grad_value = source * 3
with torch.inference_mode():
    inference_value = source * 3

print("tracked requires_grad:  ", tracked.requires_grad)
print("detached requires_grad: ", detached.requires_grad)
print("no_grad requires_grad:  ", no_grad_value.requires_grad)
print("inference requires_grad:", inference_value.requires_grad)

## 5. 가장 작은 학습 루프

아래에서는 $y=2x+1$을 따르는 데이터에서 두 parameter `w`, `b`를 찾습니다.

학습 루프의 고정 순서는 `zero_grad → forward → loss → backward → step`입니다.

In [ ]:
torch.manual_seed(7)
x_train = torch.linspace(-1, 1, 32).unsqueeze(1)
y_train = 2 * x_train + 1 + 0.05 * torch.randn_like(x_train)

linear = nn.Linear(1, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.15)

for step in range(101):
    optimizer.zero_grad(set_to_none=True)
    prediction = linear(x_train)
    loss = ((prediction - y_train) ** 2).mean()
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(f"step={step:3d} loss={loss.item():.6f}")

learned_w = linear.weight.item()
learned_b = linear.bias.item()
print(f"learned: y = {learned_w:.3f}x + {learned_b:.3f}")
assert abs(learned_w - 2.0) < 0.1 and abs(learned_b - 1.0) < 0.1

In [ ]:
# 수치 미분으로 autograd 검산하기
def f(value: torch.Tensor) -> torch.Tensor:
    return (value**3 + 2 * value).sum()


point = torch.tensor([1.5], requires_grad=True)
f(point).backward()
automatic = point.grad.detach().clone()

epsilon = 1e-4
center = point.detach()
numerical = (f(center + epsilon) - f(center - epsilon)) / (2 * epsilon)
print("autograd:", automatic.item(), "numerical:", numerical.item())
assert torch.allclose(automatic, numerical, rtol=1e-3, atol=1e-3)

## 6. Gradient를 관찰하는 디버깅 도구

hook은 backward 중 특정 tensor나 parameter의 gradient를 관찰합니다. 큰 모델에서는 모든 값을 출력하지 말고 norm, min/max, 유한성만 기록하세요.

In [ ]:
probe = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
gradient_log = {}

handles = []
for name, parameter in probe.named_parameters():

    def save_norm(grad, parameter_name=name):
        gradient_log[parameter_name] = float(grad.norm())

    handles.append(parameter.register_hook(save_norm))

batch = torch.randn(5, 4)
target = torch.tensor([0, 1, 0, 1, 1])
loss = nn.functional.cross_entropy(probe(batch), target)
loss.backward()

print("loss:", loss.item())
print("gradient norms:", gradient_log)
assert gradient_log and all(value >= 0 for value in gradient_log.values())
for handle in handles:
    handle.remove()

In [ ]:
# NaN/Inf가 생기는 지점을 빨리 찾는 작은 계약 함수
def check_tensor(
    name: str, tensor: torch.Tensor, expected_ndim: int | None = None
) -> None:
    if expected_ndim is not None:
        assert tensor.ndim == expected_ndim, (
            f"{name}: expected ndim={expected_ndim}, got {tensor.shape}"
        )
    assert torch.isfinite(tensor).all(), f"{name}: NaN 또는 Inf 발견"


healthy = torch.log(torch.tensor([1.0, 2.0, 3.0]))
check_tensor("healthy", healthy, expected_ndim=1)

unhealthy = torch.tensor([1.0, float("nan")])
try:
    check_tensor("unhealthy", unhealthy)
except AssertionError as error:
    print("검사가 문제를 잡았습니다:", error)

In [ ]:
class ShapeCheckedMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.input_dim = input_dim
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[-1] != self.input_dim:
            raise ValueError(
                f"expected [batch, {self.input_dim}], got {tuple(x.shape)}"
            )
        logits = self.network(x)
        assert logits.shape[0] == x.shape[0]
        return logits


checked_model = ShapeCheckedMLP(4, 8, 3)
print("valid output:", checked_model(torch.randn(2, 4)).shape)
try:
    checked_model(torch.randn(2, 5))
except ValueError as error:
    print("명확한 오류:", error)

## 7. 따라 치기 TODO

아래 셀은 실행 가능한 정답 형태입니다. 먼저 주석을 읽고 **새 셀에 보지 않고 다시 작성**해 보세요.

1. `[B,T,D]` 텐서에서 time 평균을 내 `[B,D]`를 만드세요.
2. padding mask `[B,T]`를 적용한 masked mean을 구현하세요.
3. 두 층 MLP의 모든 parameter에 gradient가 들어왔는지 검사하세요.

In [ ]:
sequence = torch.randn(2, 4, 3)  # [B, T, D]
attention_mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.bool)

# TODO 1: time 축 평균
plain_mean = sequence.mean(dim=1)

# TODO 2: padding을 제외한 masked mean
weights = attention_mask.unsqueeze(-1).to(sequence.dtype)
masked_mean = (sequence * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)

# TODO 3: gradient 연결 검사
exercise_model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 2))
exercise_model(masked_mean).sum().backward()
missing_grad = [name for name, p in exercise_model.named_parameters() if p.grad is None]

print("plain_mean:", plain_mean.shape, "masked_mean:", masked_mean.shape)
print("gradient가 없는 parameter:", missing_grad)
assert masked_mean.shape == (2, 3)
assert missing_grad == []

## 마무리 체크리스트

모델이 예상대로 학습되지 않을 때 다음 순서로 확인하세요.

1. 입력 한 배치의 `shape/dtype/device/range`를 출력한다.
2. 레이블 범위가 `[0, num_classes)`인지 확인한다.
3. forward 중간 shape에 `assert`를 둔다.
4. loss가 finite인지 확인한다.
5. `zero_grad → backward → step` 순서를 확인한다.
6. parameter별 `grad is None`, gradient norm을 확인한다.
7. 작은 배치 하나를 과적합할 수 있는지 시험한다.

다음 노트북에서는 이 원칙을 실제 `Dataset → DataLoader → 텍스트 분류기 → checkpoint` 파이프라인에 적용합니다.